In [1]:
import pandas as pd
import re
import contractions
import nltk
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation

DATA_PATH = "../data/comcast_consumeraffairs_complaints.csv"

In [2]:
df = pd.read_csv(DATA_PATH)

# nur Textspalte behalten
df = df[["text"]].copy()
#NaN entfernen
df = df.dropna(subset=["text"]) 
#Texttyp String
df["text"] = df["text"].astype(str)

# leere / whitespace-only Einträge und Duplikate entfernen
df = df[df["text"].str.strip().ne("")]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

df.shape

(5628, 1)

In [3]:
STOP = set(stopwords.words("english"))
# Festlegung von Stoppwörtern die erhalten bleiben sollen
NEGATIONS = {"no", "not", "nor", "never", "n't"}
STOP = STOP - NEGATIONS

# Domänenspezifische Stopwords die entfernt werden sollen
STOP |= {"comcast", "xfinity"}

# zusätzlich neu hinzugefügte Stoppwörter zur Verbesserung der Analyse
EXTRA_STOP = {"would", "said", "told", "could", "also", "really", "even", "time", "day", "hour", "minute", "one", "people","still","january","february","march","april","may","june","july","august",
         "september","october","november","december", "wa", "ha", "doe", "got", "go"}
STOP |= EXTRA_STOP

lemmatizer = WordNetLemmatizer()

url_re = re.compile(r"https?://\S+|www\.\S+")
email_re = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
num_re = re.compile(r"\b\d+([\.,]\d+)?\b")
non_letter_re = re.compile(r"[^a-zA-Z\s]+")

def preprocess(text: str) -> str:
    text = contractions.fix(text).lower()
    text = url_re.sub(" ", text)
    text = email_re.sub(" ", text)
    text = num_re.sub(" ", text)
    text = non_letter_re.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if t not in STOP]
    return " ".join(tokens)


In [4]:
df["clean_text"] = df["text"].astype(str).apply(preprocess)
df[["text", "clean_text"]].sample(3, random_state=42)

,text,clean_text
4331,After waiting on hold for more than 10 minutes...,waiting hold spoke representative explained on...
1988,I pay so much for Comcast and their so-called ...,pay much called premium channel free movie pre...
3443,I would like to give Comcast NEGATIVE stars......,like give negative star many negative star act...


In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = df["clean_text"]

bow_vec = CountVectorizer(min_df=10, max_df=0.6, ngram_range=(1,2))
tfidf_vec = TfidfVectorizer(min_df=10, max_df=0.6, ngram_range=(1,2))

X_bow = bow_vec.fit_transform(texts)
X_tfidf = tfidf_vec.fit_transform(texts)

X_bow.shape, X_tfidf.shape


((5628, 8294), (5628, 8294))

In [13]:
# --- Top Begriffe nach Gesamt-Häufigkeit (BoW) ---
terms_bow = np.array(bow_vec.get_feature_names_out())
counts = np.asarray(X_bow.sum(axis=0)).ravel()
top_bow = terms_bow[counts.argsort()[::-1][:25]]

# --- Top Begriffe nach durchschnittlicher TF-IDF (über alle Dokumente) ---
terms_tfidf = np.array(tfidf_vec.get_feature_names_out())
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_tfidf = terms_tfidf[mean_tfidf.argsort()[::-1][:25]]

print("Top BoW:", top_bow)
print("Top TF-IDF:", top_tfidf)


Top BoW: ['call' 'called' 'customer' 'get' 'internet' 'bill' 'month' 'phone'
 'cable' 'back' 'problem' 'box' 'never' 'account' 'customer service' 'new'
 'pay' 'year' 'company' 'tech' 'another' 'work' 'charge' 'technician'
 'home']
Top TF-IDF: ['call' 'bill' 'internet' 'customer' 'month' 'called' 'get' 'cable'
 'phone' 'problem' 'back' 'box' 'account' 'never' 'year'
 'customer service' 'company' 'pay' 'new' 'charge' 'tech' 'technician'
 'work' 'modem' 'week']


In [7]:
n_topics = 10
svd = TruncatedSVD(n_components=n_topics, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

terms = np.array(tfidf_vec.get_feature_names_out())

def show_lsa_topics(model, terms, topn=10):
    for i, comp in enumerate(model.components_):
        top_ids = np.argsort(np.abs(comp))[::-1][:topn]
        print(f"LSA Topic {i}: {', '.join(terms[top_ids])}")

show_lsa_topics(svd, terms, topn=10)


LSA Topic 0: call, bill, called, customer, internet, month, get, phone, cable, back
LSA Topic 1: bill, month, tech, appointment, technician, call, fee, payment, charge, pay
LSA Topic 2: speed, internet, account, channel, call, mbps, cable, called, payment, number
LSA Topic 3: box, channel, cable, speed, internet, cable box, mbps, digital, customer, dvr
LSA Topic 4: customer, modem, channel, customer service, bill, contract, speed, tech, company, credit
LSA Topic 5: bill, account, box, month, appointment, equipment, modem, tech, channel, number
LSA Topic 6: modem, cable, contract, payment, equipment, bill, package, fee, pay, get
LSA Topic 7: speed, modem, channel, mbps, called, problem, company, package, appointment, year
LSA Topic 8: customer, customer service, channel, box, phone, speed, technician, number, charge, get
LSA Topic 9: cable, technician, tech, call, internet, modem, back, get, box, account


In [8]:
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=20)
X_lda = lda.fit_transform(X_bow)

terms_b = np.array(bow_vec.get_feature_names_out())

def show_lda_topics(model, terms, topn=10):
    for i, topic in enumerate(model.components_):
        top_ids = topic.argsort()[::-1][:topn]
        print(f"LDA Topic {i}: {', '.join(terms[top_ids])}")

show_lda_topics(lda, terms_b, topn=10)


LDA Topic 0: channel, movie, watch, show, program, customer, dvr, get, account, game
LDA Topic 1: called, call, appointment, cable, supervisor, back, technician, come, customer, home
LDA Topic 2: problem, tech, phone, internet, technician, call, issue, home, work, modem
LDA Topic 3: internet, speed, cable, mbps, internet service, high, month, get, connection, modem
LDA Topic 4: customer, month, year, phone, contract, bill, customer service, fee, representative, charge
LDA Topic 5: get, customer, company, month, year, internet, like, cable, call, never
LDA Topic 6: box, cable, modem, call, channel, digital, cable box, house, signal, supervisor
LDA Topic 7: bill, month, credit, called, equipment, account, charge, pay, received, get
LDA Topic 8: account, payment, bill, pay, due, fee, paid, bank, made, late
LDA Topic 9: call, back, called, phone, get, number, box, customer, hold, someone


In [9]:
lda_top_topic = X_lda.argmax(axis=1)
lda_counts = np.bincount(lda_top_topic, minlength=n_topics)

for i in lda_counts.argsort()[::-1]:
    print(f"LDA Topic {i}: {lda_counts[i]} Dokumente")

LDA Topic 5: 1120 Dokumente
LDA Topic 2: 909 Dokumente
LDA Topic 4: 851 Dokumente
LDA Topic 7: 789 Dokumente
LDA Topic 1: 549 Dokumente
LDA Topic 9: 543 Dokumente
LDA Topic 3: 382 Dokumente
LDA Topic 8: 270 Dokumente
LDA Topic 6: 117 Dokumente
LDA Topic 0: 98 Dokumente


In [10]:
lsa_top_topic = np.abs(X_lsa).argmax(axis=1)
lsa_counts = np.bincount(lsa_top_topic, minlength=n_topics)

for i in lsa_counts.argsort()[::-1]:
    print(f"LSA Topic {i}: {lsa_counts[i]} Dokumente")


LSA Topic 0: 4957 Dokumente
LSA Topic 2: 162 Dokumente
LSA Topic 1: 128 Dokumente
LSA Topic 3: 120 Dokumente
LSA Topic 6: 65 Dokumente
LSA Topic 8: 53 Dokumente
LSA Topic 4: 49 Dokumente
LSA Topic 7: 36 Dokumente
LSA Topic 5: 33 Dokumente
LSA Topic 9: 25 Dokumente


In [11]:
terms = np.array(bow_vec.get_feature_names_out())
counts = np.asarray(X_bow.sum(axis=0)).ravel()

# nur Bigrams (enthalten ein Leerzeichen)
is_bigram = np.char.find(terms.astype(str), " ") >= 0
top_bigram_idx = np.argsort(counts[is_bigram])[::-1][:20]

print("Top Bigrams:", terms[is_bigram][top_bigram_idx])


Top Bigrams: ['customer service' 'call back' 'service not' 'internet service' 'not get'
 'phone call' 'called back' 'cable box' 'not know' 'not work'
 'phone service' 'cable internet' 'not want' 'cable service' 'service rep'
 'per month' 'service call' 'phone number' 'pay bill' 'every month']


((5628, 8294), (5628, 8294))